# upload.cptdaz.org (Front Door endpoint cptmtls-upload)


mTLS is enforced at the edge (client certificate required and validated against the CA stored in Key Vault). The origin is a private storage account (Private Link). The client sends a short-lived user delegation SAS with the upload request.

Stage 1: get a SAS. Stage 2: upload with client certificate and SAS.

Tools: bash, az, http (httpie), jq, azcopy, openssl.

## Design notes

* **mTLS at the edge:** The endpoint has `enforceMtls` enabled. The custom domain `upload.cptdaz.org` uses the mode client certificate required and validated against the CA chain in the Front Door secret `mtls-ca`, which points to a Key Vault secret. Front Door reads it with a user-assigned managed identity (Key Vault Secrets User). The Key Vault itself is private.
* **Private storage:** The origin is a private storage account reached through a Front Door private endpoint (approved with `scripts/approve-pe.sh`). Shared key and anonymous access are off, so Front Door cannot authenticate to the origin.
* **Two stages:** The client first gets a short-lived user delegation SAS (container scope, create and write only). It then uploads through Front Door with the client certificate and the SAS in the URL. mTLS decides who may reach the endpoint, the SAS decides what may be written.
* **Test PKI:** `scripts/new-pki.sh` creates a trusted CA with a valid client certificate and an untrusted CA with a second certificate. Private keys stay in `~/mtls-pki`, only `pki/ca.crt` is committed.
* **Edge cases:** Requests without a certificate or with a certificate from an untrusted CA are rejected by Front Door with HTTP 403 and never reach storage. A valid certificate without a SAS reaches storage and is rejected there.


In [ ]:
export SUB=09808f31-065f-4231-914d-776c2d6bbe34
export RG=rg-cptmtls
export DNS_SUB=ff0bb075-6c44-44ee-bb64-d46ce828c62f
export DNS_RG=domains
export DNS_ZONE=cptdaz.org
export PROFILE=afd-cptmtls-upload
export ENDPOINT=cptmtls-upload
export DOMAIN=upload-cptdaz-org
export HOST=upload.cptdaz.org
export STORAGE=stcptmtlsuploadizhwq
export CONTAINER=upload
export KV=kv-cptmtls-izhwq
export LAW=law-cptmtls
export PKI=$HOME/mtls-pki
export CERT=$PKI/client-valid.crt
export KEY=$PKI/client-valid.key
export BAD_CERT=$PKI/client-invalid.crt
export BAD_KEY=$PKI/client-invalid.key
export FILE=/tmp/upload-demo.txt
export BLOB=demo-$(date +%Y%m%d-%H%M%S).txt
export PAGER=cat
export AZCOPY_AUTO_LOGIN_TYPE=AZCLI
az account set --subscription $SUB
export LAW_ID=$(az monitor log-analytics workspace show -g $RG -n $LAW --query customerId -o tsv)
env | grep -E '^(SUB|RG|DNS_SUB|DNS_RG|DNS_ZONE|PROFILE|ENDPOINT|DOMAIN|HOST|STORAGE|CONTAINER|KV|LAW|PKI|CERT|KEY|BAD_CERT|BAD_KEY|FILE|BLOB|LAW_ID)=' | sort

## Azure account

In [ ]:
az account show --query "{name:name,id:id,user:user.name}" -o table

## Front Door configuration

In [ ]:
az afd profile show -g $RG --profile-name $PROFILE --query "{name:name,sku:sku.name,state:provisioningState}" -o table

In [ ]:
az rest --method get --url "https://management.azure.com/subscriptions/$SUB/resourceGroups/$RG/providers/Microsoft.Cdn/profiles/$PROFILE/afdEndpoints/$ENDPOINT?api-version=2026-07-01" | jq '{name, host: .properties.hostName, enforceMtls: .properties.enforceMtls, deployment: .properties.deploymentStatus}'

In [ ]:
az rest --method get --url "https://management.azure.com/subscriptions/$SUB/resourceGroups/$RG/providers/Microsoft.Cdn/profiles/$PROFILE/customDomains/$DOMAIN?api-version=2026-07-01" | jq '{host: .properties.hostName, validation: .properties.domainValidationState, deployment: .properties.deploymentStatus, mtls: .properties.mtlsSettings}'

In [ ]:
az afd route list -g $RG --profile-name $PROFILE --endpoint-name $ENDPOINT --query "[].{route:name,patterns:join(',',patternsToMatch),protocols:join(',',supportedProtocols),deployment:deploymentStatus}" -o table

In [ ]:
az afd origin list -g $RG --profile-name $PROFILE --origin-group-name og-upload-storage -o json | jq '.[] | {name, hostName, enabledState, privateLinkTarget: (.sharedPrivateLinkResource.privateLink.id | split("/")[-1])}'

## mTLS CA in Key Vault and Front Door secret

In [ ]:
az afd secret show -g $RG --profile-name $PROFILE --secret-name mtls-ca --query "{name:name,type:parameters.type,state:provisioningState}" -o table

In [ ]:
az keyvault secret show --vault-name $KV --name mtls-ca --query "{name:name,contentType:contentType,enabled:attributes.enabled}" -o table

In [ ]:
az keyvault secret show --vault-name $KV --name mtls-ca --query value -o tsv | openssl x509 -noout -subject -issuer -dates

## DNS records in the Azure DNS zone

In [ ]:
az network dns record-set cname show --subscription $DNS_SUB -g $DNS_RG -z $DNS_ZONE -n upload --query "{fqdn:fqdn,target:CNAMERecord.cname}" -o table

In [ ]:
az network dns record-set txt show --subscription $DNS_SUB -g $DNS_RG -z $DNS_ZONE -n _dnsauth.upload --query "{fqdn:fqdn,value:TXTRecords[0].value[0]}" -o table

## Private Link connection created by Front Door

In [ ]:
az network private-endpoint-connection list --id $(az storage account show -g $RG -n $STORAGE --query id -o tsv) --query "[].{name:name,status:properties.privateLinkServiceConnectionState.status,description:properties.privateLinkServiceConnectionState.description}" -o table

## Client certificates (test PKI created with scripts/new-pki.sh)

In [ ]:
openssl x509 -in $CERT -noout -subject -issuer -dates

In [ ]:
openssl x509 -in $BAD_CERT -noout -subject -issuer -dates

In [ ]:
openssl verify -CAfile ../pki/ca.crt $CERT

In [ ]:
openssl verify -CAfile ../pki/ca.crt $BAD_CERT || true

## Stage 1: user delegation SAS (container scope, create and write, 30 minutes)

The SAS is signed with a Microsoft Entra user delegation key, not with the account key (shared key access is disabled by policy).

In [ ]:
export SAS=$(az storage container generate-sas --account-name $STORAGE --name $CONTAINER --permissions cw --expiry $(date -u -d '+30 minutes' +%Y-%m-%dT%H:%MZ) --as-user --auth-mode login --https-only -o tsv)
echo $SAS | tr '&' '\n' | grep -E '^(se|sp|sr|spr)='

In [ ]:
echo "hello from httpie $(date -u)" > $FILE
cat $FILE

## Stage 2: upload through Front Door

Without a client certificate the request must be rejected at the edge.

In [ ]:
http --headers PUT "https://$HOST/$CONTAINER/$BLOB?$SAS" x-ms-blob-type:BlockBlob x-ms-version:2023-11-03 < $FILE || true

With a client certificate from an untrusted CA the request must be rejected as well.

In [ ]:
http --headers --cert=$BAD_CERT --cert-key=$BAD_KEY PUT "https://$HOST/$CONTAINER/$BLOB?$SAS" x-ms-blob-type:BlockBlob x-ms-version:2023-11-03 < $FILE || true

With a valid certificate but without SAS the storage account rejects the request.

In [ ]:
http --headers --cert=$CERT --cert-key=$KEY PUT "https://$HOST/$CONTAINER/nosas-$BLOB" x-ms-blob-type:BlockBlob x-ms-version:2023-11-03 < $FILE

With a valid certificate and the SAS the upload succeeds (`201 Created`).

In [ ]:
http --headers --cert=$CERT --cert-key=$KEY PUT "https://$HOST/$CONTAINER/$BLOB?$SAS" x-ms-blob-type:BlockBlob x-ms-version:2023-11-03 < $FILE

## Verify the blob in the private storage account

In [ ]:
az storage blob list --account-name $STORAGE --container-name $CONTAINER --auth-mode login -o json | jq '.[] | {name, size: .properties.contentLength, created: .properties.creationTime}'

In [ ]:
azcopy list "https://$STORAGE.blob.core.windows.net/$CONTAINER"

In [ ]:
azcopy copy "https://$STORAGE.blob.core.windows.net/$CONTAINER/$BLOB" /tmp/downloaded-$BLOB

In [ ]:
diff $FILE /tmp/downloaded-$BLOB && echo identical

## Reading through Front Door is denied (least privilege)

The SAS allows only create and write. The read request below passes mTLS but is rejected by Storage with `AuthorizationPermissionMismatch`.

In [ ]:
http --body --cert=$CERT --cert-key=$KEY GET "https://$HOST/$CONTAINER/$BLOB?$SAS"

## Direct upload with azcopy (bypasses Front Door, uses Entra login over the private endpoint)

In [ ]:
azcopy copy $FILE "https://$STORAGE.blob.core.windows.net/$CONTAINER/direct-$BLOB"

In [ ]:
azcopy list "https://$STORAGE.blob.core.windows.net/$CONTAINER"

## Clean up the demo blobs

In [ ]:
azcopy remove "https://$STORAGE.blob.core.windows.net/$CONTAINER/$BLOB"

In [ ]:
azcopy remove "https://$STORAGE.blob.core.windows.net/$CONTAINER/direct-$BLOB"

In [ ]:
az storage blob list --account-name $STORAGE --container-name $CONTAINER --auth-mode login --query "[].name" -o tsv

## Front Door access logs in Log Analytics

In [ ]:
az monitor log-analytics query -w $LAW_ID --analytics-query "AzureDiagnostics | where ResourceProvider == 'MICROSOFT.CDN' | where ResourceId contains toupper('$PROFILE') | summarize events=count() by Category" -o json | jq -c '.[]'

In [ ]:
az monitor log-analytics query -w $LAW_ID --analytics-query "AzureDiagnostics | where ResourceProvider == 'MICROSOFT.CDN' | where ResourceId contains toupper('$PROFILE') | top 5 by TimeGenerated desc | project TimeGenerated, Category, ResourceId" -o json | jq -c '.[]'